# Módulo 3 — Gabarito — Lista de Exercícios

Este notebook reúne os exercícios de visualização de dados com base no dataset **Olist Brazilian E-commerce**.

> Ajuste o caminho dos arquivos CSV, se necessário.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid")

# Ajuste o caminho dos arquivos, se necessário
BASE = Path("../datasets")

orders = pd.read_csv(BASE / "olist_orders_dataset.csv")
customers = pd.read_csv(BASE / "olist_customers_dataset.csv")
reviews = pd.read_csv(BASE / "olist_order_reviews_dataset.csv")
items = pd.read_csv(BASE / "olist_order_items_dataset.csv")
payments = pd.read_csv(BASE / "olist_order_payments_dataset.csv")
products = pd.read_csv(BASE / "olist_products_dataset.csv")

# Conversão de datas
orders["order_purchase_timestamp"] = pd.to_datetime(orders["order_purchase_timestamp"])
orders["order_delivered_customer_date"] = pd.to_datetime(orders["order_delivered_customer_date"], errors="coerce")

# Consolidação em um único DataFrame
df = orders.merge(customers[["customer_id", "customer_state"]], on="customer_id", how="left")
df = df.merge(reviews[["order_id", "review_score"]], on="order_id", how="left")

valor = items.groupby("order_id")["price"].sum().reset_index().rename(columns={"price": "valor"})
df = df.merge(valor, on="order_id", how="left")

pag = payments.groupby("order_id").agg(
    forma_pagamento=("payment_type", lambda x: x.mode()[0]),
    parcelas=("payment_installments", "max")
).reset_index()
df = df.merge(pag, on="order_id", how="left")

# Features temporais
df["ano_mes"] = df["order_purchase_timestamp"].dt.to_period("M").astype(str)
df["prazo_entrega"] = (df["order_delivered_customer_date"] - df["order_purchase_timestamp"]).dt.days

df.head()

## 1. Distribuição de notas de avaliação

Crie um **gráfico de barras** mostrando a **quantidade de pedidos por nota** (`review_score`, valores de 1 a 5).

**Objetivo:** entender como os clientes avaliam os pedidos.

### Gabarito

In [ ]:
# Remove nulos e conta as notas
contagem_notas = df["review_score"].dropna().astype(int).value_counts().sort_index()

# Cria o gráfico
plt.figure(figsize=(10, 6))
contagem_notas.plot(kind="bar", color=["#d62728", "#ff7f0e", "#ffbb78", "#98df8a", "#2ca02c"])

plt.title("Distribuição das notas de avaliação dos pedidos")
plt.xlabel("Nota")
plt.ylabel("Número de pedidos")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 2. Relação entre prazo de entrega e avaliação

Crie um **scatterplot** entre:

- `prazo_entrega`
- `review_score`

**Pergunta:** entregas mais rápidas recebem notas mais altas?

### Gabarito

In [ ]:
# Remove nulos e limita outliers
df_plot = df.dropna(subset=["prazo_entrega", "review_score"])
df_plot = df_plot[(df_plot["prazo_entrega"] >= 0) & (df_plot["prazo_entrega"] <= 60)]

# Amostragem para não sobrecarregar o gráfico
df_plot = df_plot.sample(min(5000, len(df_plot)), random_state=42)

plt.figure(figsize=(10, 6))
sns.scatterplot(data=df_plot, x="prazo_entrega", y="review_score", alpha=0.4)

plt.title("Relação entre prazo de entrega e nota")
plt.xlabel("Prazo de entrega (dias)")
plt.ylabel("Nota")
plt.tight_layout()
plt.show()

## 3. Distribuição do valor dos pedidos

1. Remova os valores nulos da coluna `valor`.
2. Faça um **histograma** dessa variável.

**Objetivo:** entender a faixa de preço típica dos pedidos.

### Gabarito

In [ ]:
# Cria uma cópia sem nulos e remove outliers extremos
df_plot = df.dropna(subset=["valor"]).copy()
df_plot = df_plot[df_plot["valor"] <= df_plot["valor"].quantile(0.99)]

# Cria o histograma
plt.figure(figsize=(10, 6))
sns.histplot(df_plot["valor"], bins=30, kde=True)

plt.title("Distribuição dos valores dos pedidos")
plt.xlabel("Valor (R$)")
plt.ylabel("Número de pedidos")
plt.tight_layout()
plt.show()

## 4. Valor dos pedidos por estado

Crie um **boxplot** dos valores dos pedidos (`valor`) para os **10 estados com mais pedidos**.

**Pergunta:** os valores dos pedidos variam entre estados?

### Gabarito

In [ ]:
# Seleciona os 10 estados com mais pedidos
top_estados = df["customer_state"].value_counts().head(10).index

# Remove nulos e limita outliers
df_plot = df.dropna(subset=["valor", "customer_state"])
df_plot = df_plot[df_plot["customer_state"].isin(top_estados)]
df_plot = df_plot[df_plot["valor"] <= df_plot["valor"].quantile(0.95)]

# Cria o boxplot
plt.figure(figsize=(12, 6))
sns.boxplot(data=df_plot, x="customer_state", y="valor", palette="Set3")

plt.title("Distribuição do valor dos pedidos por estado (Top 10)")
plt.xlabel("Estado")
plt.ylabel("Valor (R$)")
plt.tight_layout()
plt.show()

## 5. Composição dos status dos pedidos

Crie um gráfico de **pizza** mostrando a proporção de pedidos por `order_status`.

**Objetivo:** entender quantos pedidos foram entregues, cancelados, etc.

### Gabarito

In [ ]:
# Conta pedidos por status
contagem_status = df["order_status"].value_counts()

# Cria o gráfico de pizza
plt.figure(figsize=(8, 8))
plt.pie(
    contagem_status.values,
    labels=contagem_status.index,
    autopct="%1.1f%%",
    colors=sns.color_palette("husl", len(contagem_status)),
    startangle=90
)

plt.title("Distribuição de pedidos por status")
plt.tight_layout()
plt.show()

## 6. Evolução mensal de pedidos

Crie um **gráfico de linha** mostrando a quantidade de pedidos por mês (`ano_mes`).

**Objetivo:** visualizar a evolução temporal das vendas.

### Gabarito

In [ ]:
# Conta pedidos por mês
pedidos_mes = df.groupby("ano_mes").size().reset_index(name="num_pedidos")

# Cria o gráfico de linha
plt.figure(figsize=(12, 6))
plt.plot(pedidos_mes["ano_mes"], pedidos_mes["num_pedidos"], marker="o", linewidth=2)

plt.title("Evolução mensal do número de pedidos")
plt.xlabel("Período")
plt.ylabel("Número de pedidos")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 7. Formas de pagamento por estado

Crie um gráfico de **barras empilhadas** mostrando a composição percentual das formas de pagamento (`forma_pagamento`) nos 10 estados com mais pedidos.

### Gabarito

In [ ]:
# Seleciona os 10 estados com mais pedidos
top_estados = df["customer_state"].value_counts().head(10).index

# Remove nulos
df_plot = df.dropna(subset=["forma_pagamento", "customer_state"])
df_plot = df_plot[df_plot["customer_state"].isin(top_estados)]

# Cria tabela de frequência normalizada por linha (percentual)
composicao = pd.crosstab(
    df_plot["customer_state"],
    df_plot["forma_pagamento"],
    normalize="index"
) * 100

# Cria o gráfico empilhado
composicao.plot(kind="bar", stacked=True, figsize=(12, 6),
                color=sns.color_palette("Set2", len(composicao.columns)))

plt.title("Composição das formas de pagamento por estado")
plt.xlabel("Estado")
plt.ylabel("Percentual (%)")
plt.xticks(rotation=0)
plt.legend(title="Forma de pagamento", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()

## 8. Correlação entre variáveis

Crie um **heatmap de correlação** entre as variáveis numéricas:

- `valor`
- `prazo_entrega`
- `review_score`
- `parcelas`

**Pergunta:** quais variáveis estão mais relacionadas entre si?

### Gabarito

In [ ]:
# Seleciona as colunas numéricas
cols = ["valor", "prazo_entrega", "review_score", "parcelas"]

# Remove linhas com nulos e calcula a matriz de correlação
df_corr = df[cols].dropna()
matriz_corr = df_corr.corr()

# Cria o heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(matriz_corr, annot=True, fmt=".2f", cmap="coolwarm",
            center=0, square=True, linewidths=1, vmin=-1, vmax=1)

plt.title("Matriz de correlação entre variáveis numéricas")
plt.tight_layout()
plt.show()

## 9. Distribuição das notas por prazo de entrega

Crie faixas de prazo de entrega (`Rápido`, `Normal`, `Lento`, `Muito lento`) e um **violin plot** mostrando como as notas se distribuem em cada faixa.

### Gabarito

In [ ]:
# Cria função para categorizar o prazo
def faixa_prazo(dias):
    if pd.isna(dias):  return np.nan
    if dias <= 7:       return "1-Rápido"
    if dias <= 15:      return "2-Normal"
    if dias <= 30:      return "3-Lento"
    return "4-Muito lento"

# Aplica a função
df_plot = df.dropna(subset=["prazo_entrega", "review_score"]).copy()
df_plot["faixa"] = df_plot["prazo_entrega"].apply(faixa_prazo)

ordem = ["1-Rápido", "2-Normal", "3-Lento", "4-Muito lento"]

# Cria o violin plot
plt.figure(figsize=(12, 6))
sns.violinplot(data=df_plot, x="faixa", y="review_score",
               order=ordem, palette="RdYlGn_r")

plt.title("Distribuição das notas por faixa de prazo de entrega")
plt.xlabel("Faixa de prazo")
plt.ylabel("Nota")
plt.tight_layout()
plt.show()

## 10. Top categorias de produtos

Identifique as **10 categorias de produtos mais vendidas** (contagem de pedidos) e crie um **gráfico de barras horizontal** mostrando esse ranking.

### Gabarito

In [ ]:
# Junta a tabela de items com products para obter as categorias
items_cat = items.merge(
    products[["product_id", "product_category_name"]],
    on="product_id", how="left"
)

# Conta as top 10 categorias
top_cat = items_cat["product_category_name"].value_counts().head(10)

# Cria o gráfico
plt.figure(figsize=(12, 6))
sns.barplot(y=top_cat.index, x=top_cat.values, palette="viridis")

plt.title("Top 10 categorias de produtos mais vendidas")
plt.xlabel("Número de pedidos")
plt.ylabel("Categoria")
plt.tight_layout()
plt.show()